# Website Content Writer & Strategist

This notebook scrapes a website's content and uses the Gemini API to analyze its layout, copy, SEO, and conversion potential, generating actionable UX/branding recommendations.

## 1. Setup & Imports

Load environment variables and import required libraries.

In [4]:
from os import getenv
from dotenv import load_dotenv
from openai import OpenAI
from bs4 import BeautifulSoup
import requests

# Load environment variables from .env file
load_dotenv(override=True)

True

## 2. Configuration & Custom Exceptions

Define headers for web scraping and a custom exception class for handling API issues.

In [5]:
# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class GeminiError(Exception):
    """Exception raised when the Gemini API request fails (e.g. rate limits, invalid keys)."""
    pass

## 3. Define the Content Writer Class

This class handles fetching website content and running the Gemini analysis.

In [7]:
class content_writer:
    def __init__(self):
        pass

    def _content_writer__private_fetch_website_contents(self, url):
        """
        Return the title and contents of the website at the given url;
        truncate to 2,000 characters as a sensible limit
        """
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, "html.parser")
        title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            text = soup.body.get_text(separator="\n", strip=True)
        else:
            text = ""
        return (title + "\n\n" + text)[:2_000]

    def _content_writer__private_gemini(self, data: str):
        system_prompt = """You are the world's best Content Strategist, Conversion Copywriter, UX Writer, SEO Expert, and Brand Consultant.
          Analyze the entire website content, structure, messaging, user journey, and information hierarchy as if you were hired to elevate it to the level of industry-leading websites such as Apple, Stripe, Notion, Vercel, Airbnb, and Figma.
          Your objective is not just to rewrite content but to identify and improve anything that weakens the website's quality, credibility, engagement, clarity, user experience, SEO, branding, or conversion potential.
          For each section:
          1. Explain what is working well.
          2. Identify all content, messaging, UX writing, SEO, trust, and conversion issues.
          3. Explain why each issue matters.
          4. Provide an improved version of the content.
          5. Suggest additional content or sections that would increase trust, authority, and conversions.
          6. Recommend improvements to content hierarchy, section order, CTAs, headlines, microcopy, and storytelling.
          Additionally provide:
          - Overall website quality score (/100)
          - Content quality score (/100)
          - SEO score (/100)
          - Conversion score (/100)
          - Brand positioning score (/100)
          - Top 10 highest-impact improvements ranked by priority
          - Quick wins that can be implemented immediately
          - Strategic recommendations that would significantly improve the website
          Be brutally honest, highly specific, and actionable. Avoid generic feedback. Every recommendation must include a clear explanation and practical improvement."""
        user_prompt = f"""
      Here is the content of the website you need to analyze: 
      {data}
      """
        gemini = OpenAI(
            base_url=getenv('GEMINI_BASE_URL'),
            api_key=getenv('GEMINI_API_KEY')
        )

        try:
            response = gemini.chat.completions.create(
                model='gemini-2.5-flash',
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt}
                ]
            )
            return response.choices[0].message.content
        except Exception as e:
            # Raise our custom exception with the original exception chained
            raise GeminiError(
                "You have a gemini error. Please check the API key, quota limit, and monitor the dashboard."
            ) from e

    def main(self, url):
        data = self._content_writer__private_fetch_website_contents(url)
        return self._content_writer__private_gemini(data)

## 4. Run the Analysis

Instantiate the class and analyze a target website.

In [8]:
verify_content = content_writer()
result = verify_content.main("https://anthropic.com")
print(result)

GeminiError: You have a gemini error. Please check the API key, quota limit, and monitor the dashboard.